# walk

> The runtime of one walk step: a short tutor note, then a check: a short answer or a code task.

A walk step shows a passage, then a short note from the tutor, then one check. The journey (`journey.py`) plans the steps
and picks the check of each step. This module runs a step.

The note tells the learner what to look for in the passage. It does not answer the check. The short-answer check is a normal
tutor question, with two changes: the step decides its way and depth, and each attempt keeps the walk, the step, the section and
the page. A sitting log counts every check, also the checks that do not go through `Session.answer`.

In a code task, the learner writes one function. The tests run in a closed process with no network.

In [ ]:
#| default_exp walk

In [ ]:
#| hide
import os, tempfile
os.environ['ABHYASA_CONFIG'] = tempfile.mkdtemp()   # the settings go to a temporary folder, not to your real settings

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| export
import re, time, uuid
from fastcore.utils import *
from abhyasa.llm import for_task
from abhyasa.settings import val
from abhyasa.prompts import register, prompt
from abhyasa.tutor import Tutor, Session, public
from abhyasa.check import run_isolated
from abhyasa.nb import new_exercise
import abhyasa.journey  # adds Learning.walk_step

## The note

The prompt for the note. It gets the idea, the passage and the kind of the next check. It never gets the check question.

In [ ]:
#| export
NOTE_SP = register('walk.note', 'Walk note', 'How the tutor writes the short note under the passage of each step in a walk.',
    "You write a short note for a learner who just read a passage. Write 2 or 3 short sentences. "
    "Say what the passage says about the idea, in plain words. Do not give the answer to a question on this idea. "
    "End with what to look for next. Reply with the note only.")

The reply is cut to 3 sentences and to `walks.note` words (60 by default), even when the model writes more.

In [ ]:
#| export
def _trim(text:str, n:int=3, words:int=None) -> str:
    "The first `n` sentences of `text`, fewer when they have more than `words` words (default: the setting `walks.note`), and at most `words` words."
    words = words or val('walks.note')
    ss = re.split(r'(?<=[.!?])\s+', ' '.join(text.split()))[:n]
    while len(ss) > 1 and len(' '.join(ss).split()) > words: ss.pop()
    return ' '.join(' '.join(ss).split()[:words])

`tutor_note` writes the note once and keeps it on the step. A later call gives the kept note.

In [ ]:
#| export
def tutor_note(lr, ask, j, i:int) -> str:
    "The note for step `i` of walk `j`: kept on the step, else written from the idea and the passage."
    s = (lr.journey(j.id) or j).steps[i]
    if s.get('note'): return s['note']
    c = lr.concept(s['slug'])
    p = (f"Idea: {c.name}\n{c.get('statement', '')}\n\nThe passage:\n{(s.get('read') or {}).get('text') or 'none'}\n\n"
         f"The next check is a {(s.get('check') or {}).get('kind', 'short')} check.\n\nUse at most {val('walks.note')} words.")
    n = _trim(for_task(ask, 'hint')(p, prompt('walk.note')))
    lr.walk_step(j.id, i, note=n)
    return n

## A short answer

Each attempt of a step keeps its place in the source. `step_meta` makes that record. In a side walk, `from` names the step that opened it, as `<walk id>/<step>`.

In [ ]:
#| export
def step_meta(j, i:int) -> dict:
    "The walk, the step, the section and the page of step `i`, for the record of each attempt. A side walk adds `from`, the step that opened it."
    r = j.steps[i].get('read') or {}
    m = dict(walk=j.id, step=i, section=r.get('section'), page=r.get('page'))
    if j.get('parent'): m['from'] = '/'.join(map(str, j.parent.values()))
    return m

`open_short` opens the question of a step. The check of the step gives the way and the depth. The note goes into the question prompt. The question id stays on the step.

In [ ]:
#| export
def open_short(tutor:Tutor, j, i:int, session:str) -> AttrDict:
    "Open the short-answer check of step `i` of walk `j`, and keep its id on the step."
    s, lr = j.steps[i], tutor.learning
    ck = s['check']
    q = tutor.ask_on(lr.concept(s['slug']), ck['depth'], ck.get('why', ''), session=session, method=ck['way'],
                     meta=step_meta(j, i), read=tutor_note(lr, tutor.ask, j, i))
    lr.walk_step(j.id, i, qid=q.id)
    return q

## The sitting log

A demo or an order prediction does not go through `Session.answer`. `noted` adds one row for it, in the same shape as the rows of a question.

In [ ]:
#| export
@patch
def noted(self:Session, a:dict, floor0:int):
    "Add one log row for the attempt `a`, made outside `Session.answer` (a demo or an order prediction), and save the sitting."
    c = self.tutor.learning.concept(a['concept'])
    self.log.append(dict(qid='', concept=a['concept'], name=c.name if c else a['concept'], depth=a.get('depth', 2), why=a.get('why', ''),
                         question=a.get('question', ''), method=a.get('method', ''), correct=a.get('correct', False), outcome=a.get('outcome', ''),
                         floor0=floor0, floor=c.get('floor', 0) if c else 0, misconception=a.get('misconception', ''), turns=[], at=time.time()))
    self.save()

## A code task

The model writes the exercise: a task, a stub, a function name and the tests. `code_task` keeps the full exercise on the tutor. The page gets the question, the stub, the function name and the text of the visible tests. The solution and the hidden tests stay on the server.

In [ ]:
#| export
def code_task(tutor:Tutor, c, session:str='', meta:dict=None) -> AttrDict:
    "Open a code task on concept `c`. Return the public question, the stub, the function name and the text of the visible tests."
    r = new_exercise(tutor.ask, c)
    shown = [t if isinstance(t, str) else t['test'] for t in r.tests if isinstance(t, str) or not t.get('hidden')]
    q = AttrDict(id=uuid.uuid4().hex[:10], concept=c['slug'], name=c['name'], statement=c.get('statement', ''), subject=c.get('subject', ''),
                 floor=c.get('floor', 0), depth=4, kind='code', method='code', why='code exercise', session=session, question=r.task,
                 answer=r.solution, rubric=shown, fname=r.fname, stub=r.stub, tests=r.tests, hints=[], tries=[], meta=dict(meta or {}), at=time.time())
    q.turns = [dict(who='tutor', kind='question', text=q.question)]
    tutor.open[q.id] = q
    return AttrDict(q=AttrDict({k: v for k, v in public(q).items() if k != 'tests'}), stub=r.stub, fname=r.fname, tests=shown)

`_first_error` gives the first failing test as text. A hidden test shows only its label and its status. A visible test shows its text, the error and the value that the function gave.

In [ ]:
#| export
def _first_error(q, checks) -> str:
    "The first failing test of `checks`, as text for the learner, or an empty text."
    c = first(x for x in checks if x['status'] != 'pass')
    if c is None: return ''
    if c['hidden']: return f"{c['label']}: {c['status']}"
    t = q.tests[c['i']]
    got = f"; got {c['got']}" if c['got'] is not None else ''
    return f"This test fails: `{t if isinstance(t, str) else t['test']}` ({c['error'].rstrip(': ')}{got})"

`run_code` grades one try. A syntax error uses no try: `compile` checks the code and runs none of it. A failing try keeps the task open. A pass or the last try closes it, as `Exercise.check` does.

In [ ]:
#| export
def run_code(tutor:Tutor, qid:str, source:str, confidence:int=70) -> AttrDict:
    "Run `source` on the tests of the code task `qid` in a closed process, and grade it. A pass or the last try closes the task."
    q = tutor.open[qid]
    out = AttrDict(syntax='', checks=[], passed=0, total=len(q.tests), first_error='', tries_left=val('checks.tries') - len(q.tries), done=False)
    try: compile(source, '<code>', 'exec', dont_inherit=True)
    except (SyntaxError, ValueError) as e: return AttrDict(out, syntax=f"line {getattr(e, 'lineno', None) or 1}: {getattr(e, 'msg', None) or e}")
    r = run_isolated('assert', q.fname, source, q.tests)
    if 'checks' not in r: out.first_error = 'The code ran longer than 10 s' if r.get('timeout') else 'The code stopped before the tests ran'
    else:
        out.checks, out.passed = r['checks'], sum(c['status'] == 'pass' for c in r['checks'])
        out.first_error = _first_error(q, r['checks'])
    correct = bool(r.get('found')) and out.passed == out.total > 0
    g = AttrDict(correct=correct, grade=10 * out.passed // out.total, first_error=out.first_error, misconception='', feedback='')
    q.turns.append(dict(who='you', text=source, confidence=confidence))
    if correct: return AttrDict({**out, **tutor._close(q, True, confidence, g, answer=source)})
    q.tries.append(dict(answer=source, first_error=out.first_error, confidence=confidence))
    if len(q.tries) < (most := val('checks.tries')): return AttrDict(out, tries_left=most - len(q.tries))
    return AttrDict({**out, 'tries_left': 0, **tutor._close(q, False, q.tries[0]['confidence'], g, teach=True)})

`Session.code` runs a try on the open question of the sitting. A close goes in the sitting log.

In [ ]:
#| export
@patch
def code(self:Session, source:str, confidence:int=70) -> AttrDict:
    "Run `source` on the open code task of the sitting, log a close, and save the sitting."
    full = self.tutor.open.get(self.q.id)
    r = run_code(self.tutor, self.q.id, source, confidence)
    if r.done: self._logged(r, full)
    self.save()
    return r

## Tests for the step runtime

These tests use the small paper in `tests/walk_data.py` and a fake model.

In [ ]:
#| hide
import sys; sys.path.insert(0, '../tests'); from walk_data import *
from abhyasa.store import temp_learning
from abhyasa.llm import Fake
from abhyasa.tutor import Tutor, Session
from abhyasa.journey import journey_from_source

A walk with one fake model. It gives a note, a grade and a question.

In [ ]:
lr = temp_learning(); did = add_paper(lr)
ask = Fake({'Return JSON: {"concepts"': EXTRACT_REPLY, 'You write a short note': 'Softmax makes weights from scores. Look at what a large score does.',
            'You grade one answer': dict(correct=True, grade=9, first_error='', misconception='', feedback='Yes.')},
           dflt=dict(question='Why do the weights add to 1?', answer='Each is divided by the sum.', rubric=['divide by the sum']))
j = journey_from_source(lr, ask, did)
t = Tutor(lr, ask, context=False)

The note is kept on its step. A second call makes no model call.

In [ ]:
n = tutor_note(lr, ask, j, 1)
assert n.startswith('Softmax makes') and lr.journey(j.id).steps[1]['note'] == n
assert tutor_note(lr, Fake(), lr.journey(j.id), 1) == n

The note keeps at most 3 sentences and `walks.note` words (60 by default).

In [ ]:
long = tutor_note(lr, Fake({'You write a short note': 'One. Two. Three. Four.'}), lr.journey(j.id), 0)
assert long == 'One. Two. Three.'
wide = tutor_note(lr, Fake({'You write a short note': ' '.join(['word'] * 40) + '. ' + ' '.join(['more'] * 40) + '.'}), lr.journey(j.id), 2)
assert len(wide.split()) <= 60

The step decides the way and the depth. The question comes without its answer, and the prompt has the note.

In [ ]:
s = Session(t, journey=j.id)
lr.walk_step(j.id, 1, check=dict(way='words', kind='short', depth=2, why='plain words fit'))
q = open_short(t, lr.journey(j.id), 1, s.id); s.q = q
assert q.method == 'words' and q.depth == 2
assert 'answer' not in q and 'rubric' not in q and lr.journey(j.id).steps[1]['qid'] == q.id
assert 'The learner just read this' in ask.calls[-1][0] and n in ask.calls[-1][0]

The attempt has its walk, step, section and page. The sitting log has the close.

In [ ]:
r = s.answer('Each weight is divided by the sum of all.')
a = lr.attempts('softmax', limit=1)[0]
assert r.done and (a.walk, a.step, a.section, a.page) == (j.id, 1, '3.2.1 Scaled Dot-Product Attention', 4)
assert s.log[-1]['concept'] == 'softmax'

The review rule: the last way of a walk check is left out of the next question on the idea.

In [ ]:
lr.update('softmax', due=0)
assert all(t.ask_on(lr.concept('softmax'), 3).method != a.method for _ in range(10))

A demo or an order prediction is not made in `Session.answer`. `noted` adds its row to the sitting log and saves it.

In [ ]:
s.noted(dict(concept='softmax', question='Predict', correct=True, outcome='solid', method='demo', kind='demo'), floor0=2)
assert s.log[-1]['method'] == 'demo' and lr.session(s.id).log[-1]['method'] == 'demo'

## Tests for a code task

A new walk and a sitting. The fake model writes the softmax exercise of `tests/walk_data.py`.

In [ ]:
lr = temp_learning(); did = add_paper(lr)
ask = Fake({'Return JSON: {"concepts"': EXTRACT_REPLY, 'You write one small Python exercise': EXERCISE, 'Teach the smallest step': 'Take away the max. Check: what is softmax([5, 5])?'})
j = journey_from_source(lr, ask, did); t = Tutor(lr, ask, context=False); s = Session(t, journey=j.id)
k = code_task(t, lr.concept('softmax'), s.id, step_meta(j, 1)); s.q = k.q

The stub has no part of the solution. The page gets the two visible tests. The text of the hidden test stays on the server, and so do the answer and the rubric.

In [ ]:
assert k.fname == 'softmax' and 'm = max(x)' not in k.stub and len(k.tests) == 2
assert 'answer' not in k.q and 'rubric' not in k.q

A syntax error uses no try. The message gives the line. A `return` outside a function and a null byte are syntax errors too, though `ast.parse` accepts the first.

In [ ]:
r = s.code('def softmax(x, t=1.0)\n    return x')
assert r.syntax.startswith('line 1') and r.tries_left == 3 and not r.done
for bad in ('return 1', 'x = 1\0'): assert s.code(bad).syntax.startswith('line 1')
assert t.open[s.q.id].tries == []

The naive softmax fails on large scores. Two of three tests pass, and two tries are left. A hidden test shows no value.

In [ ]:
r = s.code(NAIVE)
assert (r.passed, r.total, r.done, r.tries_left) == (2, 3, False, 2) and 'large scores' in r.first_error
assert not any(c['got'] for c in r.checks if c['hidden'])

A correct softmax closes the task. The attempt keeps the kind, the depth, the way and the walk. The sitting log has the close, and the way `code` gets the credit.

In [ ]:
r = s.code(GOOD)
a = lr.attempts('softmax', limit=1)[0]
assert r.done and r.correct and (a.kind, a.depth, a.method, a.walk) == ('code', 4, 'code', j.id)
assert s.log[-1]['concept'] == 'softmax' and lr.stats().methods['*']['code'][0] > 0

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()